# Barra 风格因子高暴露组合（普通版）

本 notebook 直接使用米筐 Barra 风险模型 v1 的个股风格因子暴露。对每个交易日、每个风格因子，选取全 A 股中暴露最高的前 10% 股票并等权做多，比较各组合的净值走势。

这是一种因子暴露排序组合回测，并非米筐通过横截面回归计算的官方因子收益率。为避免未来函数，日期 t 的暴露只用于计算 t+1 收盘至 t+2 收盘的一日收益。

In [ ]:
# ==================== Part 0：环境与回测参数 ====================
# 本单元只负责初始化环境和集中设置口径，后续计算均引用这里的参数。

import os
import sys
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
plt.rcParams["font.sans-serif"] = ["SimHei"]
plt.rcParams["axes.unicode_minus"] = False

# 自动定位项目根目录，使 notebook 无论从根目录还是 learn 目录启动都可导入 my_utils。
PROJECT_ROOT = next(
    (path for path in [Path.cwd(), *Path.cwd().parents] if (path / "my_utils").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("未找到包含 my_utils 目录的项目根目录。")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from my_utils.rqdata import RqData
import rqdatac as rq

RqData()  # 初始化米筐；若失败，请先检查米筐账号、网络和终端授权数量。

# ---- 可修改参数 ----
START_DATE = "2024-01-01"
# 默认取昨天之前的最近交易日，避免当天收盘后数据尚未完整发布。
END_DATE = rq.get_previous_trading_date(pd.Timestamp.today().strftime("%Y-%m-%d"))
MODEL = "v1"                    # v1 对应 Barra CNE5；与参考 notebook 保持一致。
TOP_QUANTILE = 0.10              # 每个因子选择横截面暴露最高的前 10%。
BENCHMARK = "000985.XSHG"        # 中证全指，作为全市场收益参照。
PRICE_CHUNK_SIZE = 500            # 分批取价，避免一次请求过多证券导致接口失败。

STYLE_FACTORS = [
    "size",
    "beta",
    "momentum",
    "residual_volatility",
    "non_linear_size",
    "book_to_price",
    "liquidity",
    "earnings_yield",
    "growth",
    "leverage",
]
FACTOR_LABELS = {
    "size": "规模",
    "beta": "Beta",
    "momentum": "动量",
    "residual_volatility": "残差波动",
    "non_linear_size": "非线性规模",
    "book_to_price": "账面市值比",
    "liquidity": "流动性",
    "earnings_yield": "盈利收益率",
    "growth": "成长",
    "leverage": "杠杆",
}

# 交易日列表同时决定暴露日、建仓日与平仓日。至少需要三个交易日才可形成一笔收益。
trading_days = pd.DatetimeIndex(
    pd.to_datetime(rq.get_trading_dates(START_DATE, END_DATE))
).normalize()
if len(trading_days) < 3:
    raise ValueError("回测区间不足三个交易日，无法按 T+1 暴露口径计算收益。")

print("米筐初始化完成")
print("回测区间：{} ~ {}，共 {} 个交易日".format(
    trading_days[0].date(), trading_days[-1].date(), len(trading_days)
))
print("模型：{}；每个因子选择暴露最高的 {:.0%}".format(MODEL, TOP_QUANTILE))

## 回测口径

- 股票池：指定日期可交易的全 A 股普通股票；B 股代码会剔除。
- 选股：对每个 Barra 风格因子，剔除缺失值后取暴露值最高的前 10%，同日等权。
- 收益：暴露日期为 t，组合在 t+1 收盘建仓、t+2 收盘平仓；因此不会用到暴露发布前无法获得的信息。
- 基准：中证全指（000985.XSHG），使用与组合相同的 t+1 至 t+2 收盘收益。
- 简化项：不模拟涨跌停、停牌无法成交、交易费用和冲击成本。

In [ ]:
# ==================== Part 1：拉取每日暴露并构造因子股票池 ====================
# 米筐因子暴露为个股×日期的 MultiIndex 表。此处将其整理为“代码为行、因子为列”的单日截面，
# 再按每个因子的暴露排序选取前 10%。只保存选股结果，避免在内存中长期保留全部原始暴露。

def is_a_share(order_book_id):
    """排除沪深 B 股；其余 CS 类型证券作为全 A 股普通股票处理。"""
    code = str(order_book_id).split(".")[0]
    return not code.startswith(("200", "900"))


def get_active_a_share_ids(trade_date):
    """取得指定交易日可交易的全 A 股代码，确保股票池随历史上市和退市状态变化。"""
    instruments = rq.all_instruments(type="CS", date=trade_date)
    if instruments is None or instruments.empty or "order_book_id" not in instruments.columns:
        return []
    return [
        order_book_id
        for order_book_id in instruments["order_book_id"].astype(str)
        if is_a_share(order_book_id)
    ]


def normalize_exposure(raw_exposure, factors):
    """将米筐单日 MultiIndex 暴露表规范为以股票代码为索引的二维表。"""
    if raw_exposure is None or raw_exposure.empty:
        return pd.DataFrame(columns=factors)

    exposure = raw_exposure.copy()
    if isinstance(exposure.index, pd.MultiIndex):
        # 米筐文档约定索引包含 order_book_id 和 date；按名称提取可兼容两者的层级顺序。
        index_frame = exposure.index.to_frame(index=False)
        code_column = "order_book_id" if "order_book_id" in index_frame.columns else index_frame.columns[0]
        exposure.index = index_frame[code_column].astype(str).to_numpy()
        exposure.index.name = "order_book_id"
        # 单日请求理论上每个代码只有一行；保留最后一行以防接口返回重复记录。
        exposure = exposure.groupby(level=0).last()

    available_factors = [factor for factor in factors if factor in exposure.columns]
    exposure = exposure.loc[:, available_factors].apply(pd.to_numeric, errors="coerce")
    return exposure.dropna(how="all")


def select_high_exposure_stocks(exposure, factor, top_quantile):
    """对一个因子精确选出向上取整后的前 top_quantile 股票，缺失暴露不参与排序。"""
    values = exposure[factor].dropna()
    if values.empty:
        return []

    count = max(1, int(np.ceil(len(values) * top_quantile)))
    return values.nlargest(count).index.astype(str).tolist()


# t 日暴露在 t+1 后才可使用，故最后两个交易日没有完整的 t+1 至 t+2 持有期。
exposure_days = trading_days[:-2]
factor_selections = {}
selection_log = []
selected_code_set = set()

for position, exposure_date in enumerate(exposure_days, start=1):
    entry_date = trading_days[position]       # t+1：以当日收盘价格建仓
    exit_date = trading_days[position + 1]    # t+2：以当日收盘价格平仓并记收益

    try:
        universe = get_active_a_share_ids(exposure_date)
        if not universe:
            selection_log.append({
                "exposure_date": exposure_date,
                "status": "跳过",
                "reason": "当日无可交易 A 股",
            })
            continue

        raw_exposure = rq.get_factor_exposure(
            order_book_ids=universe,
            start_date=exposure_date.strftime("%Y-%m-%d"),
            end_date=exposure_date.strftime("%Y-%m-%d"),
            factors=STYLE_FACTORS,
            model=MODEL,
        )
        exposure = normalize_exposure(raw_exposure, STYLE_FACTORS)
        if exposure.empty:
            selection_log.append({
                "exposure_date": exposure_date,
                "status": "跳过",
                "reason": "米筐未返回有效因子暴露",
            })
            continue

        for factor in STYLE_FACTORS:
            if factor not in exposure.columns:
                continue

            selected_codes = select_high_exposure_stocks(exposure, factor, TOP_QUANTILE)
            if not selected_codes:
                continue

            # 以平仓日和因子作为键，后续可直接计算 entry_date 至 exit_date 的组合收益。
            factor_selections[(exit_date, factor)] = selected_codes
            selected_code_set.update(selected_codes)

        selection_log.append({
            "exposure_date": exposure_date,
            "status": "成功",
            "reason": "有效股票 {} 只".format(len(exposure)),
        })
    except Exception as error:
        # 单日接口失败不应中断整个回测；失败日期会在后续完整日期筛选中被移除。
        selection_log.append({
            "exposure_date": exposure_date,
            "status": "跳过",
            "reason": "{}: {}".format(type(error).__name__, error),
        })

    if position % 20 == 0 or position == len(exposure_days):
        print("已完成 {}/{} 个暴露日，累计选中 {} 只股票".format(
            position, len(exposure_days), len(selected_code_set)
        ))

selection_log = pd.DataFrame(selection_log).set_index("exposure_date").sort_index()
if not selected_code_set:
    raise RuntimeError("没有取得任何有效选股结果，请检查米筐权限、模型版本和日期区间。")

print("成功暴露日：{}；跳过暴露日：{}；候选股票去重后：{} 只".format(
    (selection_log["status"] == "成功").sum(),
    (selection_log["status"] == "跳过").sum(),
    len(selected_code_set),
))
display(selection_log.tail())

In [ ]:
# ==================== Part 2：取得复权价格并计算组合日收益 ====================
# 先对所有曾入选的股票一次性分批取价，再逐日计算等权组合收益，避免为每个因子重复请求价格。
# 后复权收盘价用于保留分红、送配等公司行为带来的总收益影响。

def fetch_adjusted_close(order_book_ids, start_date, end_date, chunk_size):
    """按批次下载后复权收盘价，返回日期×代码矩阵；批次失败会明确报错而不是静默补零。"""
    code_list = sorted(set(map(str, order_book_ids)))
    close_parts = []

    for start in range(0, len(code_list), chunk_size):
        code_batch = code_list[start:start + chunk_size]
        raw_price = rq.get_price(
            code_batch,
            start_date=start_date.strftime("%Y-%m-%d"),
            end_date=end_date.strftime("%Y-%m-%d"),
            frequency="1d",
            fields="close",
            adjust_type="post",
        )
        if raw_price is None or raw_price.empty:
            raise RuntimeError("价格接口未返回数据，批次起始代码为 {}".format(code_batch[0]))

        # 米筐多证券日线价格的 close 列为 MultiIndex 序列，展开后行=日期、列=代码。
        close_part = raw_price["close"].unstack("order_book_id")
        close_parts.append(close_part)

    close = pd.concat(close_parts, axis=1)
    close.index = pd.DatetimeIndex(pd.to_datetime(close.index)).normalize()
    return close.loc[:, ~close.columns.duplicated()].sort_index()


entry_start = trading_days[1]
exit_end = trading_days[-1]
close = fetch_adjusted_close(selected_code_set, entry_start, exit_end, PRICE_CHUNK_SIZE)

# 获取基准的同口径价格，基准日收益会与每个因子组合按平仓日对齐。
benchmark_raw = rq.get_price(
    BENCHMARK,
    start_date=entry_start.strftime("%Y-%m-%d"),
    end_date=exit_end.strftime("%Y-%m-%d"),
    frequency="1d",
    fields="close",
    adjust_type="post",
)
benchmark_close = benchmark_raw["close"].unstack("order_book_id").iloc[:, 0]
benchmark_close.index = pd.DatetimeIndex(pd.to_datetime(benchmark_close.index)).normalize()
benchmark_return = benchmark_close.pct_change()

# 行索引为每笔日频组合的平仓日，即 t+2。
strategy_return = pd.DataFrame(index=trading_days[2:], columns=STYLE_FACTORS, dtype=float)
holding_coverage = pd.DataFrame(index=trading_days[2:], columns=STYLE_FACTORS, dtype=float)

for exit_date in strategy_return.index:
    entry_date = trading_days[trading_days.get_loc(exit_date) - 1]

    # 停牌、退市或异常价格会导致起止价格缺失；只在当日因子组合内剔除这些股票。
    start_price = close.reindex(index=[entry_date]).iloc[0]
    end_price = close.reindex(index=[exit_date]).iloc[0]

    for factor in STYLE_FACTORS:
        selected_codes = factor_selections.get((exit_date, factor), [])
        if not selected_codes:
            continue

        stock_return = end_price.reindex(selected_codes) / start_price.reindex(selected_codes) - 1
        stock_return = stock_return.replace([np.inf, -np.inf], np.nan).dropna()
        if stock_return.empty:
            continue

        strategy_return.loc[exit_date, factor] = stock_return.mean()
        holding_coverage.loc[exit_date, factor] = len(stock_return)

# 为保证所有因子曲线在同一批真实交易日上可比较，剔除任一因子或基准缺失的日期。
common_dates = strategy_return.dropna(how="any").index.intersection(benchmark_return.dropna().index)
if len(common_dates) < 20:
    raise RuntimeError(
        "完整可比收益日不足 20 个。请缩短区间，或检查米筐暴露与价格数据权限。"
    )

strategy_return = strategy_return.loc[common_dates]
holding_coverage = holding_coverage.loc[common_dates]
benchmark_return = benchmark_return.loc[common_dates]

print("有效可比收益日：{}；{} 至 {}".format(
    len(common_dates), common_dates[0].date(), common_dates[-1].date()
))
display(holding_coverage.describe().T.rename(columns={
    "count": "有效天数", "mean": "平均持股数", "min": "最少持股数", "max": "最多持股数"
})[["有效天数", "平均持股数", "最少持股数", "max"]])

In [ ]:
# ==================== Part 3：净值曲线与绩效汇总 ====================
# 每列均为同一暴露排序规则下的独立多头组合。曲线高低仅反映该简化组合的历史表现，
# 不能等同于米筐官方隐式因子收益，也不能直接视为可实盘复制的交易策略。

portfolio_return = strategy_return.rename(columns=FACTOR_LABELS).copy()
portfolio_return["中证全指"] = benchmark_return
net_value = (1 + portfolio_return).cumprod()

fig, ax = plt.subplots(figsize=(15, 8))
for column in net_value.columns:
    line_width = 2.4 if column == "中证全指" else 1.4
    color = "black" if column == "中证全指" else None
    ax.plot(net_value.index, net_value[column], label=column, linewidth=line_width, color=color)

ax.axhline(1, color="gray", linestyle="--", linewidth=0.8)
ax.set_title("Barra 风格因子高暴露前 10% 等权多头组合净值", fontsize=14, fontweight="bold")
ax.set_ylabel("净值")
ax.grid(True, alpha=0.25)
ax.legend(ncol=2, fontsize=9)
plt.tight_layout()
plt.show()


def calculate_metrics(return_series):
    """按真实有效收益日计算常用绩效指标，不把缺失交易日误认为零收益。"""
    return_series = return_series.dropna()
    nav = (1 + return_series).cumprod()
    annual_return = nav.iloc[-1] ** (252 / len(return_series)) - 1
    annual_volatility = return_series.std(ddof=1) * np.sqrt(252)
    sharpe = (
        return_series.mean() / return_series.std(ddof=1) * np.sqrt(252)
        if return_series.std(ddof=1) > 0
        else np.nan
    )
    drawdown = nav / nav.cummax() - 1

    return pd.Series({
        "累计收益": nav.iloc[-1] - 1,
        "年化收益": annual_return,
        "年化波动": annual_volatility,
        "Sharpe": sharpe,
        "最大回撤": drawdown.min(),
        "有效收益日": len(return_series),
    })


performance = portfolio_return.apply(calculate_metrics).T.sort_values("累计收益", ascending=False)
display(performance.style.format({
    "累计收益": "{:.2%}",
    "年化收益": "{:.2%}",
    "年化波动": "{:.2%}",
    "Sharpe": "{:.2f}",
    "最大回撤": "{:.2%}",
    "有效收益日": "{:.0f}",
}))

print("提示：每个因子均按“暴露最高”做多。因子暴露的正负经济含义不同，"
      "例如 size 高暴露通常表示大市值，不能简单把所有曲线理解为传统的多空因子溢价。")

## 结果解读与边界

1. 曲线代表高暴露股票组合的收益，不等同于米筐 get_factor_return 返回的官方因子收益率。
2. 使用 t 的暴露、t+1 至 t+2 的收盘价收益，是为规避米筐暴露通常 T+1 发布带来的未来函数。
3. 本普通版未处理涨跌停、停牌不可成交、ST、交易费、冲击成本和容量约束；若用于真实资金评估，应改为订单驱动回测。
4. 可通过修改 START_DATE、END_DATE、TOP_QUANTILE 和 MODEL 调整研究范围与分组比例。